In [ ]:
# Prepare data for plotting
coords = adata_single.obsm['spatial']
celltypes = adata_single.obs['celltypes']

# Create a color map for cell types
unique_celltypes = celltypes.unique()
colors = sns.color_palette('tab20', n_colors=len(unique_celltypes))
celltype_colors = dict(zip(unique_celltypes, colors))

# Create the plot
plt.figure(figsize=(14, 12))

for celltype in unique_celltypes:
    mask = celltypes == celltype
    plt.scatter(
        coords[mask, 0],
        coords[mask, 1],
        c=[celltype_colors[celltype]],
        label=celltype,
        s=5,
        alpha=0.7,
        edgecolors='none'
    )

plt.xlabel('X Position (μm)', fontsize=12)
plt.ylabel('Y Position (μm)', fontsize=12)
plt.title(f'Spatial Distribution of Cell Types\n{selected_image[:60]}...', 
          fontsize=14, fontweight='bold')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', markerscale=3, frameon=True)
plt.axis('equal')
plt.tight_layout()
plt.show()

print("Observation: Notice how cells are organized in spatial patterns")
print("- Are tumor cells clustered?")
print("- Do immune cells penetrate tumor regions or stay at edges?")
print("- Are there distinct spatial domains?")

## Summary and Next Steps

### What We Learned

1. **Spatial Coordinates**
   - Accessed spatial positions from `.obs` and stored in `.obsm['spatial']`
   - Coordinates are in micrometers (μm) within tissue sections

2. **Spatial Visualization**
   - Created scatter plots colored by cell type
   - Highlighted specific cell types to study interactions
   - Overlaid marker expression on spatial plots

3. **Quantitative Spatial Analysis**
   - Computed distances between cell populations
   - Created density maps to identify spatial hotspots
   - Quantified spatial relationships (e.g., % tumor cells near CD8)

4. **Biological Insights**
   - Tumor-immune interfaces reveal immune infiltration patterns
   - Spatial heterogeneity within single images
   - Marker expression shows spatial gradients

### Key Concepts for Next Tutorial

These visualizations reveal that cells are not randomly distributed. In Tutorial 3, we will:

- **Build spatial neighborhood graphs** using Squidpy
- **Define neighborhoods** mathematically (Delaunay triangulation, radius-based)
- **Compute neighborhood enrichment**: which cell types prefer to be near each other?
- **Aggregate neighborhoods** to create cell-level representations

Spatial graphs formalize the concept of "neighbors" and enable systematic analysis of local microenvironments.

---

**Preparation for Tutorial 3:**
- Make sure you understand How to subset AnnData by image
- Be comfortable with spatial scatter plots
- Think about biological questions: what spatial patterns matter for cancer biology?

In [ ]:
# Create density map for CD8 T cells
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left: scatter plot
axes[0].scatter(coords[:, 0], coords[:, 1], c='lightgray', s=3, alpha=0.3)
axes[0].scatter(coords[cd8_mask, 0], coords[cd8_mask, 1], c='blue', s=10, alpha=0.7)
axes[0].set_xlabel('X Position (μm)', fontsize=11)
axes[0].set_ylabel('Y Position (μm)', fontsize=11)
axes[0].set_title('CD8 T Cells (Scatter)', fontsize=12, fontweight='bold')
axes[0].axis('equal')

# Right: 2D histogram (density map)
if cd8_mask.sum() > 10:  # Only if enough cells
    h = axes[1].hist2d(
        coords[cd8_mask, 0],
        coords[cd8_mask, 1],
        bins=50,
        cmap='Blues',
        cmin=1
    )
    plt.colorbar(h[3], ax=axes[1], label='CD8 Cell Count')
    axes[1].set_xlabel('X Position (μm)', fontsize=11)
    axes[1].set_ylabel('Y Position (μm)', fontsize=11)
    axes[1].set_title('CD8 T Cell Density', fontsize=12, fontweight='bold')
    axes[1].axis('equal')

plt.tight_layout()
plt.show()

print("The density map reveals local concentrations of CD8 T cells")
print("Dark blue regions = high CD8 density (potential immune hotspots)")

### Spatial Density Maps

We can create density maps to see where specific cell types are concentrated.

In [ ]:
from scipy.spatial.distance import cdist

# Get coordinates of tumor cells and CD8 cells
tumor_coords = coords[tumor_mask]
cd8_coords = coords[cd8_mask]

if len(tumor_coords) > 0 and len(cd8_coords) > 0:
    # Compute pairwise distances
    distances = cdist(tumor_coords, cd8_coords, metric='euclidean')
    
    # For each tumor cell, find distance to nearest CD8 cell
    min_distances = distances.min(axis=1)
    
    # Visualize distribution
    plt.figure(figsize=(10, 6))
    plt.hist(min_distances, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
    plt.xlabel('Distance to Nearest CD8 T Cell (μm)', fontsize=12)
    plt.ylabel('Number of Tumor Cells', fontsize=12)
    plt.title('Distribution of Tumor-CD8 Distances', fontsize=14, fontweight='bold')
    plt.axvline(min_distances.median(), color='red', linestyle='--', 
                label=f'Median: {min_distances.median():.1f} μm')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"Statistics of tumor-CD8 distances:")
    print(f"  Mean: {min_distances.mean():.1f} μm")
    print(f"  Median: {min_distances.median():.1f} μm")
    print(f"  Min: {min_distances.min():.1f} μm")
    print(f"  Max: {min_distances.max():.1f} μm")
    
    # What percentage ofchter tumor cells have a CD8 within 50 μm?
    within_50um = (min_distances <= 50).sum() / len(min_distances) * 100
    print(f"\nTumor cells with CD8 within 50μm: {within_50um:.1f}%")
else:
    print("Not enough tumor or CD8 cells in this image for distance analysis")

### Computing Pairwise Distances

Let's compute the distance between each tumor cell and its nearest CD8 T cell.

## Part 4: Quantifying Spatial Distribution

Beyond visualization, we can quantify spatial patterns.

In [ ]:
# Student workspace - Task 1
# TODO: Select a different image and visualize it


# Student workspace - Task 2
# TODO: Highlight Tumor and MacCD163 cells


# Student workspace - Task 3
# TODO: Visualize Ki67 expression spatially

### Exercise: Student Exploration

**Task 1:** Choose a different image from the dataset (image index 5-10) and create a spatial plot colored by cell type.

**Task 2:** For your chosen image, highlight tumor cells and macrophages (MacCD163). Do they co-localize?

**Task 3:** Create a spatial plot showing Ki67 (proliferation marker) expression. Where are proliferating cells located?

In [ ]:
# Select a marker to visualize
marker_to_plot = 'PD1'
marker_idx = np.where(adata_single.var['marker'] == marker_to_plot)[0][0]

# Get expression values
expr_values = adata_single.layers['exprs_arcsinh'][:, marker_idx]

# Create spatial plot colored by expression
plt.figure(figsize=(14, 12))
scatter = plt.scatter(
    coords[:, 0],
    coords[:, 1],
    c=expr_values,
    cmap='viridis',
    s=10,
    alpha=0.7,
    edgecolors='none'
)
plt.colorbar(scatter, label=f'{marker_to_plot} Expression (arcsinh)')
plt.xlabel('X Position (μm)', fontsize=12)
plt.ylabel('Y Position (μm)', fontsize=12)
plt.title(f'Spatial Distribution of {marker_to_plot} Expression', fontsize=14, fontweight='bold')
plt.axis('equal')
plt.tight_layout()
plt.show()

print(f"Mean {marker_to_plot} expression: {expr_values.mean():.3f}")
print(f"Max {marker_to_plot} expression: {expr_values.max():.3f}")
print("\nInterpretation: High PD1 expression indicates exhausted or dysfunctional T cells")

### Visualizing Marker Expression Spatially

Let's plot the spatial distribution of PD1 (immune checkpoint marker) expression.

In [ ]:
# Add arcsinh-transformed expression layer if not already present
if 'exprs_arcsinh' not in adata_single.layers:
    adata_single.layers['exprs_arcsinh'] = np.arcsinh(adata_single.layers['exprs'] / 5)

# Get marker names
marker_names = adata_single.var['marker'].values
print(f"Available markers: {marker_names[:10]} ...")

## Part 3: Marker Expression in Space

We can overlay protein marker expression on spatial plots to see where specific markers are expressed.

In [ ]:
# Highlight Tumor and CD8 T cells
fig, ax = plt.subplots(figsize=(14, 12))

# Plot all other cells in gray (background)
other_mask = ~adata_single.obs['celltypes'].isin(['Tumor', 'CD8'])
ax.scatter(
    coords[other_mask, 0],
    coords[other_mask, 1],
    c='lightgray',
    s=3,
    alpha=0.3,
    label='Other cells',
    edgecolors='none'
)

# Highlight Tumor cells
tumor_mask = adata_single.obs['celltypes'] == 'Tumor'
ax.scatter(
    coords[tumor_mask, 0],
    coords[tumor_mask, 1],
    c='red',
    s=10,
    alpha=0.6,
label='Tumor',
    edgecolors='none'
)

# Highlight CD8 T cells
cd8_mask = adata_single.obs['celltypes'] == 'CD8'
ax.scatter(
    coords[cd8_mask, 0],
    coords[cd8_mask, 1],
    c='blue',
    s=10,
    alpha=0.6,
    label='CD8 T cells',
    edgecolors='none'
)

ax.set_xlabel('X Position (μm)', fontsize=12)
ax.set_ylabel('Y Position (μm)', fontsize=12)
ax.set_title('Tumor-Immune Interface: Tumor vs CD8 T cells', fontsize=14, fontweight='bold')
ax.legend(markerscale=2)
ax.axis('equal')
plt.tight_layout()
plt.show()

print(f"Tumor cells: {tumor_mask.sum()}")
print(f"CD8 T cells: {cd8_mask.sum()}")
print("\nQuestion: Do CD8 T cells infiltrate tumor regions or remain at the periphery?")

### Highlighting Specific Cell Types

Sometimes we want to focus on interactions between specific cell types (e.g., tumor cells and CD8 T cells).

### Creating a Spatial Scatter Plot

Now we'll plot cells in 2D space, colored by their cell type.

In [ ]:
# Select the first image
selected_image = all_images[0]
print(f"Selected image: {selected_image}")

# Subset the AnnData object
adata_single = adata[adata.obs['image'] == selected_image].copy()

print(f"\nSingle image contains {adata_single.n_obs} cells")
print(f"Cell type distribution:")
print(adata_single.obs['celltypes'].value_counts())

### Selecting and Subsetting a Single Image

We'll select the first image and create a subset of the AnnData object containing only cells from that image.

In [ ]:
# List all available images
all_images = adata.obs['image'].unique()
print(f"Total images: {len(all_images)}")
print("\nFirst 10 images:")
for i, img in enumerate(all_images[:10], 1):
    n_cells = (adata.obs['image'] == img).sum()
    print(f"{i}. {img[:50]}... ({n_cells} cells)")

## Part 2: Visualizing a Single Tissue Image

Let's start by visualizing one image to understand tissue architecture.

In [ ]:
# Extract spatial coordinates from .obs and store in .obsm
adata.obsm['spatial'] = adata.obs[['Pos_X', 'Pos_Y']].values

print("Spatial coordinates stored in .obsm['spatial']")
print(f"Shape: {adata.obsm['spatial'].shape}")
print("\nFirst 5 cells' coordinates:")
print(adata.obsm['spatial'][:5])

### Extracting Spatial Coordinates

Squidpy and other spatial analysis tools expect coordinates in `.obsm['spatial']`. Let's prepare this.

In [ ]:
# Load the dataset
DATA_PATH = 'data/train_adata.h5ad'
adata = anndata.read_h5ad(DATA_PATH)

print(f"Loaded dataset with {adata.n_obs} cells and {adata.n_vars} markers")
print(f"Images: {adata.obs['image'].nunique()}")
print(f"Cell types: {adata.obs['celltypes'].nunique()}")

In [ ]:
# Import libraries
import anndata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization settings
plt.rcParams['figure.figsize'] = (12, 8)
sns.set_style('white')

print("Libraries imported successfully!")

## Part 1: Loading Data and Preparing Spatial Coordinates

# Tutorial 2: Spatial Coordinates and Tissue Visualization

**Systems Biology - Spatial Proteomics Module**  
**University of Warsaw, Faculty of Mathematics, Informatics and Mechanics**

---

## Learning Objectives

By the end of this tutorial, you will be able to:

1. Access and interpret spatial coordinates in AnnData objects
2. Visualize tissue architecture with cells colored by cell type
3. Explore spatial patterns in individual images
4. Identify regions of interest (e.g., tumor-immune interfaces)
5. Quantify spatial distributions within images
6. Prepare data for spatial neighborhood analysis

---

## Biological Context

### From Composition to Architecture

In Tutorial 1, we analyzed images based on their **cell type composition** (what cell types are present and in what proportions). This is a 0th-order analysis that ignores spatial information.

Now we move to **spatial architecture**: not just *what* cells are present, but *where* they are located. This matters because:

- **Immune-tumor interface**: CD8 T cells must be near tumor cells to kill them
- **Tertiary lymphoid structures**: B cells and T cells cluster together  
- **Exclusion patterns**: Some tumors exclude immune cells to tumor edges
- **Neighborhood effects**: Cell behavior depends on local microenvironment

### Spatial Coordinates in IMC

Each cell has:
- **Pos_X**: X coordinate in micrometers within the tissue section
- **Pos_Y**: Y coordinate in micrometers
- **image**: Which tissue image the cell belongs to

Coordinates are stored in `.obs` and can be extracted to `.obsm['spatial']` for analysis.